In [2]:
"""
Class activation topography (CAT) for EEG model visualization, combining class activity map and topography
Code: Class activation map (CAM) and then CAT

refer to high-star repo on github: 
https://github.com/WZMIAOMIAO/deep-learning-for-image-processing/tree/master/pytorch_classification/grad_cam

Salute every open-source researcher and developer!
"""


import argparse
import os
gpus = [1]
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ["CUDA_VISIBLE_DEVICES"] = ','.join(map(str, gpus))
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import numpy as np
import math
import glob
import random
import itertools
import datetime
import time
import datetime
import sys
from scipy import io

import torchvision.transforms as transforms
from torchvision.utils import save_image, make_grid

from torch.utils.data import DataLoader
from torch.autograd import Variable
from torchsummary import summary
import torch.autograd as autograd


import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.nn.init as init

from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms
from sklearn.decomposition import PCA

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch import nn
from torch import Tensor
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from einops import rearrange, reduce, repeat
from einops.layers.torch import Rearrange, Reduce
# from common_spatial_pattern import csp
from Model.GPoolingwithdecoderForAnalysis2.model_zhengjiao import Net
from Model.GPoolingwithdecoderForAnalysis2.args_zjBCIsingletrain_val import *
import matplotlib.pyplot as plt
from torch.backends import cudnn
# from tSNE import plt_tsne
# from grad_cam.utils import GradCAM, show_cam_on_image
from utils.visualize_utils import GradCAM, show_cam_on_image

from torchviz import make_dot
cudnn.benchmark = False
cudnn.deterministic = True

model = Net(num_channels=data_config.num_channel, 
                    len_window=data_config.len_window, 
                    d_model=data_config.d_model, 
                    frame_stride=data_config.frame_stride,
                    num_frame=data_config.num_frame,
                    num_head=data_config.num_head, 
                    encoder_num_layers=data_config.encoder_num_layers, 
                    low_p = data_config.low_p,
                    dropout=data_config.dropout, 
                    transformerparwiseforward_dimrat=data_config.transformerparwiseforward_dimrat, 
                    statenum=data_config.statenum,
                    num_class=data_config.num_class)

device = torch.device("cpu")
# data = np.load('./grad_cam/train_data.npy')  
# print(np.shape(data))


In [ ]:
x = torch.randn(2, 22, 438)

# 前向传播
all_att, out, dec_x, y, states = model(x)

# 生成计算图
dot = make_dot(dec_x, params=dict(model.named_parameters()))

# 保存计算图
dot.render('model_graph1', format='png')

In [3]:
nSub = 1
target_category = 2  # set the class (class activation mapping)
def reshape_transform(tensor):
    result = rearrange(tensor, 'b (h w) e -> b e (h) (w)', h=1)
    return result

path = 'C:/2023Experiment/ON2024-04-08At22-44-06/ckpl/'+'Fold'+str(nSub).zfill(2)+'/'
cpkl_path = os.path.join(path, 'Conv+TransFormer+GPoolingV2+ClassTransHead+decoder+正交约束_best_params.pkl')
net_dict = torch.load(cpkl_path,map_location=device)
net_dict = net_dict['net_state_dict']
model.load_state_dict(net_dict)




<All keys matched successfully>

In [ ]:
def hook_fn(module, input, output):
    # 把output存储起来
    features.append(output)

def compute_similarity(feature_maps):
    feature_maps = feature_maps.detach().numpy()
    num_features = feature_maps.shape[1]  # 获取通道数
    similarity_matrix = np.zeros((num_features, num_features))
    
    for i in range(num_features):
        for j in range(i, num_features):
            if i != j:
                # 计算两个特征图的皮尔逊相关系数
                corr = np.corrcoef(feature_maps[0, i].flatten(), feature_maps[0, j].flatten())[0, 1]
                similarity_matrix[i, j] = corr
                similarity_matrix[j, i] = corr  # 相似性矩阵是对称的
    return similarity_matrix

# 初始化模型和数据
dummy_input = torch.randn(1, 3, 28, 28)  # 假设输入是1x3x28x28的图片

# 存储特征的列表
features = []

# 给指定层添加hook
model.conv2.register_forward_hook(hook_fn)

# 运行模型
output = model(dummy_input)

# 计算特征图的相似性矩阵
if features:
    similarity_matrix = compute_similarity(features[0])
    print("Similarity Matrix:\n", similarity_matrix)


In [ ]:
target_layers = [model.Embedding]

In [ ]:
cam = GradCAM(model=model, target_layers=target_layers, use_cuda=False, reshape_transform=reshape_transform)
import mne
from matplotlib import mlab as mlab

biosemi_montage = mne.channels.make_standard_montage('biosemi64')
index = [37, 9, 10, 46, 45, 44, 13, 12, 11, 47, 48, 49, 50, 17, 18, 31, 55, 54, 19, 30, 56, 29]  # for bci competition iv 2a
biosemi_montage.ch_names = [biosemi_montage.ch_names[i] for i in index]
biosemi_montage.dig = [biosemi_montage.dig[i+3] for i in index]
info = mne.create_info(ch_names=biosemi_montage.ch_names, sfreq=160., ch_types='eeg')




In [ ]:
statebasis = model.classhead.summary_token
rearrange(statebasis, 'b f n -> b n f')
statebasis = torch.autograd.Variable(statebasis, requires_grad=True)
grayscale_cam = cam(input_tensor=statebasis)

In [ ]:

print(model.classhead.summary_token.size())